# Воспроизведение финального решения

Здесь запускается тот же pipeline, что и в командной строке: E5-small и USER2-base ищут кандидатов, CatBoost отбирает итоговые 50. Описание признаков, проверки качества и ограничений — в README.

Перед запуском установите зависимости, скачайте закреплённые encoder-модели командой из README и положите входные Parquet в data/. Для полного benchmark требуется CUDA GPU и время на кодирование каталога.


In [ ]:
import os, sys, platform, importlib
from pathlib import Path
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'  # обход OpenMP в проверенной Windows среде
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
import torch
ROOT = Path.cwd()
if not (ROOT / 'configs' / 'final.json').exists():
    ROOT = ROOT.parent
if not (ROOT / 'configs' / 'final.json').exists():
    raise FileNotFoundError('Запустите ноутбук из корня проекта или из notebooks/')
sys.path.insert(0, str(ROOT))
print('Python:', platform.python_version(), 'CUDA:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0), 'VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
for package in ('torch', 'transformers', 'catboost', 'numpy', 'pandas', 'scipy', 'pyarrow'):
    print(package, importlib.import_module(package).__version__)


## Пути

Значения по умолчанию указывают на benchmark. Для небольшого теста можно задать переменные `AVITO_QUERIES`, `AVITO_ITEMS`, `AVITO_OUTPUT`, `AVITO_CACHE` до запуска ядра. Веса encoder'ов и модель CatBoost задаёт `configs/final.json`; их наличие проверяется до predict. Кеш необязателен: `AVITO_CACHE=''` означает пересчёт без сохранения.


In [ ]:
import json
QUERIES = Path(os.getenv('AVITO_QUERIES', ROOT / 'data' / 'benchmark_queries.parquet'))
ITEMS = Path(os.getenv('AVITO_ITEMS', ROOT / 'data' / 'benchmark_items.parquet'))
OUTPUT = Path(os.getenv('AVITO_OUTPUT', ROOT / 'answer_reproduced.csv'))
CONFIG = ROOT / 'configs' / 'final.json'
CACHE = os.getenv('AVITO_CACHE', '') or None
config = json.loads(CONFIG.read_text(encoding='utf8'))
print('queries:', QUERIES, 'items:', ITEMS, 'output:', OUTPUT, 'cache:', CACHE)
for name, entry in config['encoders'].items():
    folder = ROOT / entry['local_path']
    if not (folder / 'config.json').exists():
        raise FileNotFoundError(f'Нет весов {name}: {folder}; см. README')


## Загрузка и проверка схемы

ID остаются строками. Ошибки обязательных колонок, типов и дубликатов выдаёт общий модуль `solution.predict`.


In [ ]:
import pandas as pd
from solution.predict import validate_tables, predict
queries, items = validate_tables(pd.read_parquet(QUERIES), pd.read_parquet(ITEMS))
print('Запросов:', len(queries), 'объявлений:', len(items))
print('Категория 0:', int(queries.search_category.eq(0).sum()))


## Общий inference pipeline

Эта ячейка выполняет загрузку локальных моделей, embeddings, поиск, BM25 и CatBoost. Она вызывает ту же функцию, что и команда `python -m solution.predict`; отдельной реализации здесь нет.


In [ ]:
from time import perf_counter
start = perf_counter()
result_path = predict(QUERIES, ITEMS, OUTPUT, CONFIG, cache_dir=CACHE)
print('CSV:', result_path, 'время, с:', round(perf_counter() - start, 1))


## Проверка CSV и примеры

Проверяется состав ответа по входному корпусу; локальная разметка скрытого benchmark не используется.


In [ ]:
import hashlib
from IPython.display import display
answer = pd.read_csv(result_path, dtype=str, keep_default_na=False)
assert answer.columns.tolist() == ['query_id', 'answer']
assert set(answer.query_id) == set(queries.query_id)
assert not answer.query_id.duplicated().any()
valid_ids = set(items.item_id)
for value in answer.answer:
    ids = value.split()
    assert len(ids) == min(50, len(items))
    assert len(ids) == len(set(ids)) and set(ids) <= valid_ids
print('Проверено запросов:', len(answer))
print('SHA-256:', hashlib.sha256(Path(result_path).read_bytes()).hexdigest())
display(answer.head(3))


In [ ]:
lookup = items.set_index('item_id')
for row in answer.head(3).itertuples(index=False):
    print('query_id:', row.query_id)
    print('query:', queries.loc[queries.query_id.eq(row.query_id), 'search_query'].iloc[0])
    sample = lookup.loc[row.answer.split()[:5], ['item_title_raw', 'item_infm_params_text']].copy()
    sample['item_infm_params_text'] = sample['item_infm_params_text'].fillna('').astype(str).str[:100]
    print(sample.to_string())
    print()
